# File Finder
Notebook with a helper that reads `boilest.db` and prints video file paths for each directory listed in the `directories` table.

In [1]:
import sqlite3
import os
from pathlib import Path

def scan_directories_from_db(db_path='boilest.db', extensions=None):
    """Read `directories` table from `db_path` and yield video file paths as found.
    Args:
        db_path (str|Path): path to sqlite database file.
        extensions (iterable): file extensions to consider as video files.
    """
    if extensions is None:
        extensions = ['.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv', '.ts']

    db_path = Path(db_path)
    if not db_path.exists():
        print(f'Database not found: {db_path}')
        return

    conn = sqlite3.connect(str(db_path))
    cur = conn.cursor()
    try:
        cur.execute("SELECT path FROM directories")
        rows = cur.fetchall()
    except Exception as e:
        print('Error reading directories table:', e)
        rows = []
    finally:
        conn.close()

    for (path,) in rows:
        path = os.path.expanduser(path)
        if not os.path.isdir(path):
            print(f'Directory not found: {path}')
            continue
        # Walk directory and yield matching files as they are found
        for root, dirs, files in os.walk(path):
            for file in files:
                for ext in extensions:
                    if file.lower().endswith(ext.lower()):
                        yield os.path.join(root, file)

# Example usage: iterate over generator:
# for fp in scan_directories_from_db('boilest.db'):
#     print(fp)

In [2]:
# Invoke the scanner using the default database path
# Print each result as it is yielded
for fp in scan_directories_from_db():
    print(fp)

/media\Media 1\test_file_01.mp4
/media\Media 1\Media A\test_file_02.mp4
/media\Media 1\Media A\test_file_03.mp4
/media\Media 2\test_file_04.mp4
/media\Media 2\test_file_05.mp4
/media\Media 3\test_file_06.mp4
/media\Media 4\test.mkv
/media\Media 4\test_file_07.MP4


# Probing
Probe using ffprobe for codec details

In [26]:
import subprocess
import json
import shutil
from pathlib import Path

def probe_file(filepath, timeout=30, ffprobe_path=None):
    """Run ffprobe on `filepath`, print and return the raw JSON output from ffprobe.
    Uses `-loglevel quiet -show_entries ... -of json` to limit returned fields.
    Returns the parsed JSON (dict) or None on error.
    """
    filepath = Path(filepath)
    if not filepath.exists():
        print(f'File not found: {filepath}')
        return None

    # Use explicit ffprobe_path if provided, otherwise try to locate it
    if ffprobe_path:
        ffprobe_exec = str(ffprobe_path)
    else:
        ffprobe_exec = shutil.which('ffprobe')
    if not ffprobe_exec:
        print('ffprobe not found in PATH or provided path')
        return None

    # Use the requested ffprobe flags to limit output fields
    entries = 'format:stream=index,codec_type,codec_name,channel_layout,format=nb_streams'
    cmd = [ffprobe_exec, '-loglevel', 'quiet', '-show_entries', entries, '-of', 'json', str(filepath)]
    print('one')
    try:
        print('two')
        cp = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, check=True)
        data = json.loads(cp.stdout)
        print('three')
    except subprocess.CalledProcessError as e:
        print('ffprobe failed:', e)
        return None
    except json.JSONDecodeError as e:
        print('Failed to parse ffprobe JSON output:', e)
        return None
    except Exception as e:
        print('Error running ffprobe:', e)
        return None

    # Print full ffprobe JSON output and return it
    #print(json.dumps(data, indent=2))
    return data

# Example:
# probe_file('path/to/video.mp4')
# probe_file('path/to/video.mp4', ffprobe_path=r'C:\full\path\to\ffprobe.exe')

In [27]:
info =probe_file('/media/Media 1/test_file_01.mp4')

one
two
three


In [ ]:
def evaluate_streams_for_encoding(probe_data, desired_codecs=None, default_allow=True, target_codecs=None, output_template='output.mp4'):
    """Decide which streams should be encoded and build an ffmpeg command.

    Args:
        probe_data (dict): parsed ffprobe JSON output (from `probe_file`).
        desired_codecs (dict): mapping of codec_type -> list of allowed codec_name strings.
            Example: {'video': ['h264','hevc'], 'audio': ['aac','ac3']}
        default_allow (bool): if True, stream types not present in `desired_codecs` are treated as allowed (no encode).
        target_codecs (dict): mapping of codec_type -> ffmpeg encoder name to use when encoding.
            Example: {'video':'libx264', 'audio':'aac', 'subtitle':'copy'}
        output_template (str): suggested output filename (returned in command). Use '{input}' or replace later.

    Returns:
        dict: {
            'streams': [ {index,.., should_encode, reason} ... ],
            'ffmpeg_cmd': list_of_command_tokens
        }
    """
    if probe_data is None:
        return {'streams': [], 'ffmpeg_cmd': None}
    if desired_codecs is None:
        desired_codecs = {
            'video': ['h264', 'hevc', 'vp9', 'av1'],
            'audio': ['aac', 'mp3', 'ac3', 'opus'],
            'subtitle': ['subrip']
        }
    if target_codecs is None:
        target_codecs = {
            'video': 'libx264',
            'audio': 'aac',
            'subtitle': 'copy'
        }

    streams = probe_data.get('streams', [])
    decisions = []

    # We'll build lists of codec choices per stream type in stream-order (as ffmpeg expects)
    video_codecs = []
    audio_codecs = []
    subtitle_codecs = []

    # counters to track per-type stream positions
    v_idx = a_idx = s_idx = 0

    for s in streams:
        index = s.get('index')
        codec_type = s.get('codec_type')
        codec_name = (s.get('codec_name') or '').lower()
        should_encode = False
        reason = 'ok'

        if codec_type in desired_codecs:
            allowed = [c.lower() for c in desired_codecs.get(codec_type, [])]
            if allowed:
                if codec_name not in allowed:
                    should_encode = True
                    reason = f"codec '{codec_name}' not in allowed list for {codec_type}"
                else:
                    should_encode = False
                    reason = 'codec allowed'
            else:
                should_encode = True
                reason = f"no allowed codecs listed for {codec_type}; encoding"
        else:
            if default_allow:
                should_encode = False
                reason = f"no config for {codec_type}; skipping encode (default_allow=True)"
            else:
                should_encode = True
                reason = f"no config for {codec_type}; encoding (default_allow=False)"

        # Decide codec to put in ffmpeg command for this stream
        if codec_type == 'video':
            if should_encode:
                codec = target_codecs.get('video', 'libx264')
            else:
                codec = 'copy'
            video_codecs.append(codec)
            v_idx += 1
        elif codec_type == 'audio':
            if should_encode:
                codec = target_codecs.get('audio', 'aac')
            else:
                codec = 'copy'
            audio_codecs.append(codec)
            a_idx += 1
        elif codec_type in ('subtitle', 'closed_captions'):
            if should_encode:
                codec = target_codecs.get('subtitle', 'copy')
            else:
                codec = 'copy'
            subtitle_codecs.append(codec)
            s_idx += 1
        else:
            # unknown stream types: copy by default unless should_encode
            codec = 'copy' if not should_encode else target_codecs.get(codec_type, 'copy')

        decisions.append({
            'index': index,
            'codec_type': codec_type,
            'codec_name': codec_name,
            'should_encode': should_encode,
            'reason': reason
        })

    # Build ffmpeg command tokens. Use input placeholder '{input}' to be replaced by caller.
    cmd = ['ffmpeg', '-hide_banner', '-y', '-i', '{input}', '-map', '0']

    # Add per-stream codec options using ffmpeg specifiers
    for i, c in enumerate(video_codecs):
        # ffmpeg supports -c:v[:stream_index] codec
        if i == 0:
            spec = '-c:v'
        else:
            spec = f'-c:v:{i}'
        cmd.extend([spec, c])

    for i, c in enumerate(audio_codecs):
        if i == 0:
            spec = '-c:a'
        else:
            spec = f'-c:a:{i}'
        cmd.extend([spec, c])

    for i, c in enumerate(subtitle_codecs):
        if i == 0:
            spec = '-c:s'
        else:
            spec = f'-c:s:{i}'
        cmd.extend([spec, c])

    # Output placeholder
    cmd.append(output_template)

    return {'streams': decisions, 'ffmpeg_cmd': cmd}


# Example usage:
# info = probe_file('/path/to/file.mp4')
# result = evaluate_streams_for_encoding(info, desired_codecs={'video':['h264'], 'audio':['aac']}, target_codecs={'video':'libx264','audio':'aac'})
# print(result['streams'])
# print(' '.join(result['ffmpeg_cmd']).replace('{input}','/path/to/file.mp4'))

In [29]:
# Example usage:
info = probe_file('/media/Media 1/test_file_01.mp4')
decisions = evaluate_streams_for_encoding(info, desired_codecs={'video':['av1'], 'audio':['aac']})
print(decisions)

one
two
three
[{'index': 0, 'codec_type': 'video', 'codec_name': 'h264', 'should_encode': True, 'reason': "codec 'h264' not in allowed list for video"}]


In [ ]:
import shutil
import subprocess

def execute_ffmpeg_command(ffmpeg_cmd, input_path, ffmpeg_path=None, timeout=3600, capture_output=False):
    """Execute an ffmpeg command token list produced by `evaluate_streams_for_encoding`."""
    # Args:
    #   ffmpeg_cmd (list): list of command tokens (may contain '{input}' placeholder).
    #   input_path (str|Path): path to input file used to replace '{input}'.
    #   ffmpeg_path (str): optional explicit path to ffmpeg executable.
    #   timeout (int): timeout in seconds.
    #   capture_output (bool): if True, capture stdout/stderr and return them; otherwise stream to notebook output.

    # Normalize tokens and input
    tokens = list(ffmpeg_cmd) if isinstance(ffmpeg_cmd, (list, tuple)) else str(ffmpeg_cmd).split()
    input_str = str(input_path)

    # Replace placeholder occurrences
    tokens = [t.replace('{input}', input_str) for t in tokens]

    # Resolve ffmpeg executable
    if ffmpeg_path:
        ffmpeg_exec = str(ffmpeg_path)
    else:
        ffmpeg_exec = shutil.which('ffmpeg')
    if not ffmpeg_exec:
        print('ffmpeg not found in PATH or provided path')
        return None

    # If first token is 'ffmpeg', replace with full path
    if tokens and tokens[0].lower() == 'ffmpeg':
        tokens[0] = ffmpeg_exec

    try:
        if capture_output:
            cp = subprocess.run(tokens, capture_output=True, text=True, timeout=timeout)
        else:
            cp = subprocess.run(tokens, text=True, timeout=timeout)
    except subprocess.TimeoutExpired as e:
        print('ffmpeg timed out:', e)
        return None
    except Exception as e:
        print('Error running ffmpeg:', e)
        return None

    if cp.returncode != 0:
        if capture_output:
            print('ffmpeg failed: returncode', cp.returncode)
            print('stdout:', cp.stdout)
            print('stderr:', cp.stderr)
        else:
            print('ffmpeg exited with code', cp.returncode)
    return cp

# Convenience wrapper that takes probe info and runs encoding:
def encode_file_with_decisions(probe_data, input_path, desired_codecs=None, target_codecs=None, output_template=None, ffmpeg_path=None, timeout=3600, capture_output=False):
    res = evaluate_streams_for_encoding(probe_data, desired_codecs=desired_codecs, target_codecs=target_codecs, output_template=(output_template or '{input}.encoded.mp4'))
    cmd = res.get('ffmpeg_cmd')
    if cmd is None:
        print('No ffmpeg command generated')
        return None
    return execute_ffmpeg_command(cmd, input_path, ffmpeg_path=ffmpeg_path, timeout=timeout, capture_output=capture_output)

# Encoding
